# Taller 1 — Metaheurísticas: programación semanal de quirófanos

Este notebook usa `cirugias.py` (constructivo, búsqueda local VND, GRASP y LNS) y `exacto.py`
(modelo entero con HiGHS) para resolver las instancias de la carpeta `InstanciasQuirofanos`.

**Estructura**
1. Configuración y carga de instancias
2. Diagnóstico de cada instancia (tamaño, holgura, cuellos de botella)
3. Punto 1 — Constructivo, sobre una instancia de ejemplo
4. Punto 3 — Búsqueda local (VND), GRASP y LNS
5. Experimento sobre todas las instancias y comparación con el óptimo
6. Visualización de la programación
7. Punto 2 — Análisis de la junta clínica
8. Exportar resultados

**Objetivo (lexicográfico):** (1) maximizar cirugías programadas, (2) minimizar la
dispersión $\sum_{dep}(\text{zonas usadas}-1)$, (3) desempate: minimizar cirugías fuera
del ala mayoritaria de su departamento.

## 1. Configuración y carga de instancias

In [ ]:
import glob, importlib, json, os, random, time
from collections import Counter, defaultdict

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# Recarga los scripts por si se reemplazaron con el kernel abierto
import cirugias; importlib.reload(cirugias)
from cirugias import (Instancia, Solucion, cargar_instancia, constructivo, busqueda_local,
                      grasp, lns, validar, cota_superior)

try:
    import highspy
except ImportError:
    highspy = None
    print("highspy no está instalado en este kernel: pip install highspy (y reinicia el kernel)")

HAY_EXACTO = False
if highspy is not None:
    import exacto; importlib.reload(exacto)
    faltan = [f for f in ("exacto", "exacto_lexico", "constructivo_aleatorio") if not hasattr(exacto, f)]
    if faltan:
        print(f"Tu exacto.py ({exacto.__file__}) es una versión vieja: le falta {faltan}. Reemplázalo por la nueva.")
    else:
        from exacto import exacto, exacto_lexico, constructivo_aleatorio
        HAY_EXACTO = True

pd.set_option("display.max_columns", 50)

# ---------------- parámetros del experimento
CARPETA      = "InstanciasQuirofanos"
GRASP_ITERS  = 20      # iteraciones GRASP por corrida
GRASP_ALPHA  = 0.3     # tamaño de la RCL (0 = voraz puro, 1 = aleatorio)
SEMILLAS     = [1, 2, 3, 4, 5]   # corridas independientes de GRASP y LNS
LNS_ITERS    = 200     # iteraciones destruir-reparar por corrida
LNS_K        = 4       # cirugías retiradas en cada destrucción
T_LIM_EXACTO = 120     # segundos por instancia para HiGHS
print("Modelo exacto disponible:", HAY_EXACTO)

In [ ]:
rutas = sorted(glob.glob(os.path.join(CARPETA, "**", "*.json"), recursive=True))
assert rutas, f"No se encontraron .json en {CARPETA}/"
INST = {os.path.splitext(os.path.basename(p))[0]: Instancia(cargar_instancia(p)) for p in rutas}
print(f"{len(INST)} instancias cargadas:", ", ".join(INST))

## 2. Diagnóstico de las instancias

- **Holgura** = cirugías / huecos de sala (|R|·|Days|·|Blocks|). Si es baja, el reto real es la dispersión, no el conteo.
- **Cota superior**: cirugías con al menos una opción factible, acotada por la capacidad de salas.
- **Cuellos de botella por tipo**: cuántos cirujanos y salas atienden cada tipo, frente a cuántas cirugías lo requieren.

In [ ]:
filas = []
for nom, ins in INST.items():
    huecos = len(ins.R) * len(ins.Days) * len(ins.Blocks)
    filas.append(dict(instancia=nom, cirugias=len(ins.S), salas=len(ins.R), cirujanos=len(ins.K),
                      departamentos=len(ins.Dept_Surgeon), huecos_sala=huecos,
                      ocupacion_min=round(len(ins.S) / huecos, 2), cota_sup=cota_superior(ins)))
resumen = pd.DataFrame(filas).set_index("instancia")
resumen

In [ ]:
def cuellos_de_botella(ins):
    # Por tipo de cirugía: demanda vs. oferta de cirujanos y salas (en bloques).
    tipos = sorted(set(ins.type_of.values()))
    nb = len(ins.Blocks)
    out = []
    for t in tipos:
        cs = [s for s in ins.S if ins.type_of[s] == t]
        ks = sorted({k for s in cs for k in ins.surg_for[s]})
        rs = sorted({r for s in cs for r in ins.rooms_for[s]})
        out.append(dict(tipo=t, cirugias=len(cs), cirujanos=len(ks), salas=len(rs),
                        bloques_cirujano=sum(len(ins.days_k[k]) * nb for k in ks),
                        lista_cirujanos=", ".join(ks), lista_salas=", ".join(rs)))
    return pd.DataFrame(out).set_index("tipo")

for nom, ins in INST.items():
    print(f"=== {nom}")
    display(cuellos_de_botella(ins))

## 3. Punto 1 — Constructivo

Sobre una instancia de ejemplo. La regla de selección toma primero la cirugía **más restringida**
(la que tiene menos opciones factibles en ese momento). Para ella elige la opción que menos
**presiona** los recursos que otras cirugías pendientes necesitan, con una penalización si la sala
cae en un ala nueva para el departamento.

In [ ]:
EJEMPLO = list(INST)[0]          # cambia aquí la instancia a inspeccionar
ins = INST[EJEMPLO]

t = time.time(); sol_c = constructivo(ins); t_c = time.time() - t
print(f"{EJEMPLO}: programadas {sol_c.n_prog()}/{len(ins.S)} | dispersión {sol_c.dispersion()} "
      f"| minoritarias {sol_c.minoritarias()} | {t_c*1000:.1f} ms")
print("Violaciones:", validar(sol_c) or "ninguna")

In [ ]:
def tabla_solucion(sol):
    ins = sol.ins; filas = []
    for s in ins.S:
        o = sol.x[s]
        if o is None:
            filas.append(dict(cirugia=s, tipo=ins.type_of[s], estado="SIN PROGRAMAR"))
            continue
        d, b, r, k = o
        filas.append(dict(cirugia=s, tipo=ins.type_of[s], dia=d, bloque=b, sala=r,
                          zona=ins.zone_of[r], cirujano=k, departamento=ins.dept_of.get(k),
                          estado="ok"))
    df = pd.DataFrame(filas)
    orden_d = {d: i for i, d in enumerate(ins.Days)}
    return df.sort_values(by=["dia", "bloque", "sala"],
                          key=lambda c: c.map(orden_d) if c.name == "dia" else c,
                          na_position="last").reset_index(drop=True)

tabla_solucion(sol_c)

## 4. Punto 3 — Búsqueda local (VND), GRASP y LNS

Vecindarios del VND, en orden:

| | Movimiento | Mejora |
|---|---|---|
| N1 | Insertar una cirugía no programada en un hueco libre | +1 cirugía |
| N2 | Cadena de expulsión, profundidad 1 | +1 cirugía |
| N3 | Cadena de expulsión, profundidad 2 | +1 cirugía |
| N4 | Sacar 1 cirugía programada y meter 2 no programadas | +1 cirugía |
| N5 | Reubicar una cirugía (día/bloque/sala/cirujano) | dispersión |
| N6 | Intercambiar salas de dos cirugías del mismo día y bloque | dispersión |

Las cadenas de expulsión (N2–N3) exigen reinsertar **todas** las cirugías expulsadas, así que
nunca cambian *cuáles* cirugías quedan por fuera. N4 cubre ese caso. Aun así, el VND queda
atrapado en óptimos locales en las instancias más ajustadas, y para eso están dos estrategias:

- **GRASP:** reinicia desde constructivos aleatorizados (RCL con α).
- **LNS (destruir y reparar):** parte de la solución del VND. Retira hasta *k* cirugías que
  ocupan los recursos que necesita una cirugía no programada, repara con el mismo constructivo
  y aplica VND. Acepta si no empeora, lo que le permite moverse por mesetas.

Para aislar el aporte de la búsqueda local, también se parte de un constructivo **aleatorio**.

In [ ]:
sol_ls = busqueda_local(sol_c.copia())
print(f"Constructivo + LS : {sol_ls.n_prog()} cirugías, dispersión {sol_ls.dispersion()}")

if HAY_EXACTO:
    sol_al = constructivo_aleatorio(ins, random.Random(7))
    n0, d0 = sol_al.n_prog(), sol_al.dispersion()
    busqueda_local(sol_al)
    print(f"Aleatorio         : {n0} cirugías, dispersión {d0}  ->  + LS: "
          f"{sol_al.n_prog()} cirugías, dispersión {sol_al.dispersion()}")

sol_g = grasp(ins, GRASP_ITERS, GRASP_ALPHA, seed=1)
print(f"GRASP + LS        : {sol_g.n_prog()} cirugías, dispersión {sol_g.dispersion()}")
sol_n = lns(sol_ls, LNS_ITERS, LNS_K, seed=1)
print(f"Constr + LS + LNS : {sol_n.n_prog()} cirugías, dispersión {sol_n.dispersion()}")
assert not validar(sol_g) and not validar(sol_n)

## 5. Experimento sobre todas las instancias

Para cada instancia: constructivo, constructivo + LS, GRASP + LS y LNS (estos dos con varias
semillas: mejor y promedio).
Si hay solver, se compara con el **óptimo exacto** (máximo de cirugías y, con ese número fijo,
mínima dispersión).

In [ ]:
resultados, MEJORES = [], {}
for nom, ins in INST.items():
    fila = dict(instancia=nom, cirugias=len(ins.S))
    t = time.time(); c = constructivo(ins); fila["t_constr_s"] = time.time() - t
    fila["constr_n"], fila["constr_disp"] = c.n_prog(), c.dispersion()

    t = time.time(); l = busqueda_local(c.copia()); fila["t_ls_s"] = time.time() - t
    fila["ls_n"], fila["ls_disp"] = l.n_prog(), l.dispersion()

    corridas, t = [], time.time()
    for sd in SEMILLAS:
        g = grasp(ins, GRASP_ITERS, GRASP_ALPHA, seed=sd)
        assert not validar(g), validar(g)
        corridas.append(g)
    fila["t_grasp_s"] = (time.time() - t) / len(SEMILLAS)
    mejor_g = max(corridas, key=lambda s: s.objetivo())
    fila["grasp_n_mejor"], fila["grasp_disp_mejor"] = mejor_g.n_prog(), mejor_g.dispersion()
    fila["grasp_n_prom"] = sum(s.n_prog() for s in corridas) / len(corridas)

    corridas_l, t = [], time.time()
    for sd in SEMILLAS:
        n_ = lns(l, LNS_ITERS, LNS_K, seed=sd)
        assert not validar(n_), validar(n_)
        corridas_l.append(n_)
    fila["t_lns_s"] = (time.time() - t) / len(SEMILLAS)
    mejor_l = max(corridas_l, key=lambda s: s.objetivo())
    fila["lns_n_mejor"], fila["lns_disp_mejor"] = mejor_l.n_prog(), mejor_l.dispersion()
    fila["lns_n_prom"] = sum(s.n_prog() for s in corridas_l) / len(corridas_l)

    mejor = max([mejor_g, mejor_l], key=lambda s: s.objetivo())
    MEJORES[nom] = mejor

    if HAY_EXACTO:
        t = time.time()
        n_opt, st = exacto(ins, T_LIM_EXACTO)
        _, d_opt, st2 = exacto_lexico(ins, n_opt, T_LIM_EXACTO)
        fila.update(opt_n=n_opt, opt_disp=d_opt, t_exacto_s=time.time() - t,
                    estado_solver=st.split(".")[-1])
        fila["gap_%"] = 100 * (n_opt - mejor.n_prog()) / max(n_opt, 1)
    resultados.append(fila)
    print(f"{nom:20} listo")

res = pd.DataFrame(resultados).set_index("instancia")
res.round(3)

In [ ]:
cols = ["cirugias", "constr_n", "ls_n", "grasp_n_mejor", "lns_n_mejor"] + (["opt_n"] if HAY_EXACTO else []) \
     + ["constr_disp", "ls_disp", "grasp_disp_mejor", "lns_disp_mejor"] \
     + (["opt_disp", "gap_%"] if HAY_EXACTO else [])
tabla_informe = res[cols]
tabla_informe

In [ ]:
# Brecha frente al óptimo por método (0 = óptimo). Si no hay solver, se grafica el conteo.
METODOS = {"Constructivo": ("constr_n", "#2a78d6"), "Constr. + LS": ("ls_n", "#eb6834"),
           "GRASP + LS": ("grasp_n_mejor", "#1baf7a"), "LS + LNS": ("lns_n_mejor", "#eda100")}
x = list(range(len(res))); w = 0.2
fig, ax = plt.subplots(figsize=(max(7, 1.1 * len(res)), 4))
for i, (lab, (col, colr)) in enumerate(METODOS.items()):
    y = res["opt_n"] - res[col] if HAY_EXACTO else res[col]
    ax.bar([j + (i - 1.5) * w for j in x], y, width=w - 0.03, color=colr, label=lab)
ax.set_xticks(x); ax.set_xticklabels(res.index, rotation=30, ha="right")
ax.set_ylabel("Cirugías por debajo del óptimo" if HAY_EXACTO else "Cirugías programadas")
ax.set_title("Brecha frente al óptimo por método" if HAY_EXACTO else "Cirugías programadas por método",
             loc="left", pad=30)
if HAY_EXACTO:
    ax.yaxis.get_major_locator().set_params(integer=True)
ax.axhline(0, color="#222", lw=1)
ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="y", alpha=0.25)
ax.legend(frameon=False, ncol=4, loc="lower left", bbox_to_anchor=(0, 1.0), fontsize=9)
plt.tight_layout(); plt.show()

## 6. Visualización de la programación

Cada celda es una sala en un día y bloque, coloreada por el departamento del cirujano.
Las salas se agrupan por ala (zona): una buena solución deja cada color concentrado en un solo bloque de filas.

In [ ]:
DEP_COL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]

def grafico_programacion(sol, titulo=""):
    ins = sol.ins
    salas = sorted(ins.R, key=lambda r: (ins.zone_of[r], r))
    slots = [(d, b) for d in ins.Days for b in ins.Blocks]
    deps = sorted(set(ins.dept_of.values()))
    color_dep = {d: DEP_COL[i % len(DEP_COL)] for i, d in enumerate(deps)}
    fig, ax = plt.subplots(figsize=(1.0 * len(slots) + 1.5, 0.5 * len(salas) + 1.4))
    for i, r in enumerate(salas):
        for j, (d, b) in enumerate(slots):
            s = sol.room.get((r, d, b))
            if s is None:
                ax.add_patch(plt.Rectangle((j, i), 0.94, 0.9, color="#ececec"))
                continue
            k = sol.x[s][3]
            ax.add_patch(plt.Rectangle((j, i), 0.94, 0.9, color=color_dep[ins.dept_of.get(k)]))
            ax.text(j + 0.47, i + 0.45, f"{s}\n{k}", ha="center", va="center", fontsize=7, color="white")
    ax.set_xlim(0, len(slots)); ax.set_ylim(len(salas), 0)
    ax.set_xticks([j + 0.47 for j in range(len(slots))])
    ax.set_xticklabels([f"{d}\n{b}" for d, b in slots], fontsize=8)
    ax.set_yticks([i + 0.45 for i in range(len(salas))])
    ax.set_yticklabels([f"{r} ({ins.zone_of[r]})" for r in salas], fontsize=8)
    # separadores de zona
    for i in range(1, len(salas)):
        if ins.zone_of[salas[i]] != ins.zone_of[salas[i - 1]]:
            ax.axhline(i - 0.05, color="#222", lw=1.5)
    ax.tick_params(length=0); [sp.set_visible(False) for sp in ax.spines.values()]
    ax.legend(handles=[Patch(color=color_dep[d], label=d) for d in deps], frameon=False,
              ncol=len(deps), loc="lower left", bbox_to_anchor=(0, 1.0), fontsize=8)
    ax.set_title(f"{titulo}  —  {sol.n_prog()} cirugías, dispersión {sol.dispersion()}",
                 loc="left", pad=24, fontsize=10)
    plt.tight_layout(); plt.show()

grafico_programacion(sol_c, f"{EJEMPLO}: constructivo")
grafico_programacion(MEJORES[EJEMPLO], f"{EJEMPLO}: mejor solución (GRASP / LNS)")

## 7. Punto 2 — Junta clínica

Cada departamento necesita un día libre común para su junta, en el que (idealmente) todos sus
cirujanos estén disponibles. Para cada departamento se calculan:

- **días posibles**: la intersección de la disponibilidad de sus cirujanos;
- **costo de capacidad** de cada día: cuántas cirugías pierden opciones si ese día se reserva,
  y si alguna queda sin ninguna opción (cirugía *bloqueada*).

Una estrategia simple es fijar en cada departamento el día de menor costo, quitárselo a sus
cirujanos y correr el mismo constructivo + LS. Así se mide cuánto cuesta la junta en cirugías.

In [ ]:
def analisis_junta(ins):
    filas = []
    for dep, ks in ins.Dept_Surgeon.items():
        comunes = set(ins.Days)
        for k in ks:
            comunes &= ins.days_k[k]
        for d in ins.Days:
            if d not in comunes:
                continue
            afectadas = bloqueadas = 0
            for s in ins.S:
                ok = [(dd, k) for dd in ins.days_s[s] for k in ins.surg_for[s] if dd in ins.days_k[k]]
                sin = [(dd, k) for dd, k in ok if not (dd == d and k in ks)]
                afectadas += len(sin) < len(ok)
                bloqueadas += bool(ok) and not sin
            filas.append(dict(departamento=dep, dia=d, cirugias_afectadas=afectadas,
                              cirugias_bloqueadas=bloqueadas))
        if not comunes:
            filas.append(dict(departamento=dep, dia="(ninguno común)"))
    return pd.DataFrame(filas)

def aplicar_junta(D, dias_junta):
    # Devuelve una copia de la instancia con el día de junta retirado a cada cirujano.
    D = json.loads(json.dumps(D))
    for dep, d in dias_junta.items():
        for k in D["Dept_Surgeon"][dep]:
            D["Days_Surgeon"][k] = [x for x in D["Days_Surgeon"][k] if x != d]
    return D

filas_junta = []
for nom, p in zip(INST, rutas):
    ins = INST[nom]
    aj = analisis_junta(ins)
    validos = aj.dropna(subset=["cirugias_afectadas"])
    elegidos = (validos.sort_values(["cirugias_bloqueadas", "cirugias_afectadas"])
                       .groupby("departamento").first()["dia"].to_dict())
    ins_j = Instancia(aplicar_junta(cargar_instancia(p), elegidos))
    sol_j = grasp(ins_j, GRASP_ITERS, GRASP_ALPHA, seed=1)
    sol_j = max([sol_j, lns(sol_j, LNS_ITERS, LNS_K, seed=1)], key=lambda z: z.objetivo())
    filas_junta.append(dict(instancia=nom, dias_junta=elegidos,
                            sin_junta=MEJORES[nom].n_prog(), con_junta=sol_j.n_prog(),
                            disp_con_junta=sol_j.dispersion()))
    if nom == EJEMPLO:
        print(f"Análisis de días posibles para {nom}:")
        display(aj)

pd.DataFrame(filas_junta).set_index("instancia")

## 8. Exportar resultados

In [ ]:
os.makedirs("resultados", exist_ok=True)
res.to_csv("resultados/resumen.csv")
for nom, sol in MEJORES.items():
    tabla_solucion(sol).to_csv(f"resultados/{nom}_programacion.csv", index=False)
print("Guardado en ./resultados/:", sorted(os.listdir("resultados")))